In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pickle

from sbi import utils as utils
from sbi import analysis as analysis
from sbi.inference.posteriors import DirectPosterior

In [ ]:
import sys
sys.path.append('../src')

from simulator import smFRET_simulator_wrapper_from_config
from priors import generate_posterior_prior_from_config
from utils import eval_spline_from_config

In [ ]:
data_cfg = "/home/dingeldein/Desktop/fret_sbi/smFRET_sbi/data/base/config.yaml"

In [ ]:
prior = generate_posterior_prior_from_config(data_cfg, device="cuda")
eval_spline = eval_spline_from_config(data_cfg)

In [ ]:
def bin_along_time(x, n_bins=100):
    n = len(x)
    n_per_bin = int(n / n_bins)
    x_binned = np.zeros(n_bins)
    for i in range(n_bins):
        x_binned[i] = np.sum(x[i*n_per_bin:(i+1)*n_per_bin])
    return x_binned

In [ ]:
fret_simulator = smFRET_simulator_wrapper_from_config(data_cfg)

In [ ]:
true_params = torch.tensor([ 0.300, 2.0938, -1.0873, 1.05556,  -1.05016, 2.01840,  8.0000, 9.000, 0.0500,  0.0500,  0.9500,  0.9500,  0.9000,  0.9000,  0.1000,  0.1000], dtype=torch.float64)

In [ ]:
x, y = eval_spline(
    params=true_params,
    x_eval=np.linspace(2, 16, 1000),
    N_knots=5,
    x_min=4,
    x_max=14,
    boundary_condition='bound',
    max_G0=30,
    spline_type="cubic"
)
plt.plot(x, y)
plt.ylim(-5, 5)

In [ ]:
observation = torch.stack([fret_simulator(true_params) for _ in range(1)])

In [ ]:
observation.sum()

In [ ]:
idx = 0
plt.figure(figsize=(7, 2.5))
plt.plot(bin_along_time(observation[idx, :,0].numpy(), n_bins=200), label="Donor")
plt.plot(bin_along_time(observation[idx, :,1].numpy(), n_bins=200), label="acceptor")
plt.ylabel("Photon count")
plt.xlabel("Time")


In [ ]:
with open("/home/dingeldein/Desktop/fret_sbi/smFRET_sbi/outputs/density_estimator/2026-03-07/23-29-47/density_estimator.pkl", "rb") as f:
    density_estimator = pickle.load(f)
density_estimator = density_estimator.cuda()

In [ ]:
posterior = DirectPosterior(density_estimator, prior=prior)
samples = posterior.sample((10000,), x=observation.cuda(), max_sampling_batch_size=1000).cpu()

In [ ]:
posterior.set_default_x(observation.cuda())
map_estimate = posterior.map(
    num_iter=1_000,
    num_to_optimize=1,
    learning_rate=0.01,
    init_method="posterior",
    num_init_samples=5,
    save_best_every=10,
    show_progress_bars=False,
    force_update=False,
)

In [ ]:
_ = analysis.pairplot(
    samples,
    figsize=(7,7),
    points=true_params.cpu(),
    points_offdiag={'markersize': 6},
    points_colors='r', 
)

In [ ]:
samples[:, 1:] = samples[:, 1:] - samples[:, 1:].mean(dim=1, keepdim=True)
map_estimate[1:] = map_estimate[1:] - map_estimate[1:].mean()
true_params[1:5] = true_params[1:5] - true_params[1:5].mean()

In [ ]:
for i in range(1000):
    x, y = eval_spline(
        params=samples[i],
        x_eval=np.linspace(3, 15, 1000),
    )
    plt.plot(x, y, color='gray', alpha=0.01)
x, y = eval_spline(
    params=map_estimate[0].cpu(),
    x_eval=np.linspace(3, 15, 1000),
)
plt.plot(x, y, color='black', label='MAP Estimate')
x, y = eval_spline(
    params=true_params,
    x_eval=np.linspace(3, 15, 1000),
)
plt.plot(x, y+0.5, color='r', label='Ground Truth')
plt.xlabel("Distance (nm)", fontsize=16)
plt.ylabel(r"$\Delta G [k_{B}T]$", fontsize=16)
plt.ylim(-5, 5)
plt.grid()
plt.legend(fontsize=13)

In [ ]:
plt.hist(samples[:, 0].numpy(), bins=50, density=True)
plt.vlines(true_params[0], ymin=0, ymax=plt.gca().get_ylim()[1], color='r', label='Ground Truth')
plt.xlabel("log(D)", fontsize=16)
plt.ylabel("p(log(D)|x)", fontsize=16)